# **Notebook Walkthrough: Source M - Cook Political Partisan Voting Index (PVI)**

This notebook prepares the **Source M** dataset—Cook Political Partisan Voting Index (PVI)—used in the Bridge Grades methodology to measure the partisan lean of each U.S. congressional district relative to the national average.

- **Source M:** *Cook Political Partisan Voting Index (PVI)*  
- **Origin:** Obtained via paid subscription from [Cook Political](https://www.cookpolitical.com/cook-pvi)  
- **Date downloaded:** April 03, 2025 (Data updates periodically. Last checked: February 16)

The PVI values generated here will be merged into the master district-level dataset for final Bridge Grade calculations.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import seaborn as sns
import matplotlib.pyplot as plt
import glob

import warnings
import pandas as pd
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=(SettingWithCopyWarning))

### **Load & Parse PVI Values**

We read the Excel sheet, split the `"2025 Cook PVI"` column into **party** and **numeric** components, and cast the numeric part to integer.

In [3]:
# Load raw Cook PVI for the 119th Congress (2025–26)
house_pvi = pd.read_excel(
    '/content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2026_06_02/Raw_M_2026_06_02/Cook PVI 1997-2025.xlsx',
    sheet_name='119 (25-26)'
)

In [4]:
# Split "2025 Cook PVI" into Party and Number
house_pvi[['PVI_Party','PVI_Number']] = (
    house_pvi['2025 Cook PVI']
    .str.split('+', n=1, expand=True)
)

In [5]:
# Convert numeric part to integer, coerce errors to NaN
house_pvi['PVI_Number'] = pd.to_numeric(house_pvi['PVI_Number'], errors='coerce').fillna(0).astype(int)

In [6]:
house_pvi.head(10)

,State,Number,Member,Party,2025 Cook PVI,PVI_Party,PVI_Number
0,Alabama,1,Barry Moore,R,R+27,R,27
1,Alabama,2,Shomari Figures,D,D+5,D,5
2,Alabama,3,Mike Rogers,R,R+23,R,23
3,Alabama,4,Robert Aderholt,R,R+33,R,33
4,Alabama,5,Dale Strong,R,R+15,R,15
5,Alabama,6,Gary Palmer,R,R+20,R,20
6,Alabama,7,Terri Sewell,D,D+13,D,13
7,Alaska,AL,Nick Begich,R,R+6,R,6
8,Arizona,1,David Schweikert,R,R+1,R,1
9,Arizona,2,Eli Crane,R,R+7,R,7


### **Load & Parse Senate PVI**

We read the CSV of state‐level PVI, split the `"Raw PVI"` into party lean and numeric components, and convert the numeric part to integer.

In [7]:
# Load raw Senate PVI by state
senate_pvi = pd.read_csv(
    '/content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2026_06_02/Raw_M_2026_06_02/2025 PVI States.csv'
)

In [8]:
# Split "2025 PVI" into Lean Party and Number
senate_pvi[['PVI_Party','PVI_Number']] = (
    senate_pvi['2025 PVI']
    .str.split('+', n=1, expand=True)
)

In [9]:
# Convert numeric part to integer, default 0 on errors
senate_pvi['PVI_Number'] = pd.to_numeric(
    senate_pvi['PVI_Number'], errors='coerce'
).fillna(0).astype(int)

In [10]:
senate_pvi.head(10)

,State,2025 PVI,Raw PVI,Rank (D to R),PVI_Party,PVI_Number
0,Alabama,R+15,R+14.81,44,R,15
1,Alaska,R+6,R+6.46,32,R,6
2,Arizona,R+2,R+2.06,27,R,2
3,Arkansas,R+15,R+15.31,46,R,15
4,California,D+12,D+11.52,6,D,12
5,Colorado,D+6,D+5.96,14,D,6
6,Connecticut,D+8,D+8.08,8,D,8
7,Delaware,D+8,D+8.01,9,D,8
8,D.C.,D+44,D+43.6,1,D,44
9,Florida,R+5,R+5.39,29,R,5


### **Prepare party-state-district keys**

We add a key that will later be used to identify the different parties. We also need a common union key with the format “XX-DD” (for example, “CA-12”). We:

1. Add a key to identify the party with the initial of its name.

2. Assign the full names of the states to the USPS abbreviations in *both* DataFrames.

3. Create a district string with zero padding, treating “At Large” as district 1.

In [11]:
# Read in the 119th Congress data with bioguide ids
files = sorted(glob.glob('/content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2026_06_02/Raw_Others_2026_06_02/119th_Congress_20260602.csv'))
latest = files[-1]
df_119 = pd.read_csv(latest)
meta_data = df_119[["Name", "bioguide_id", "Party", "Chamber", "State", "District"]]

# meta_data["Party"] is R, if the party is Republican, D if Democrat, I if Independent
meta_data["Party_Abbr"] = meta_data["Party"].replace({
    "Republican": "R",
    "Democratic": "D",
    "Independent": "I"
})

In [12]:
meta_data.head(10)

,Name,bioguide_id,Party,Chamber,State,District,Party_Abbr
0,Laura Friedman,F000483,Democratic,House,California,30.0,D
1,Matt Van Epps,V000139,Republican,House,Tennessee,7.0,R
2,Cory A. Booker,B001288,Democratic,Senate,New Jersey,NaN,D
3,Adam B. Schiff,S001150,Democratic,Senate,California,NaN,D
4,Alan Armstrong,A000383,Republican,Senate,Oklahoma,NaN,R
5,Analilia Mejia,M001246,Democratic,House,New Jersey,11.0,D
6,Clay Fuller,F000485,Republican,House,Georgia,14.0,R
7,Joaquin Castro,C001091,Democratic,House,Texas,20.0,D
8,Kristen McDonald Rivet,M001237,Democratic,House,Michigan,8.0,D
9,Jason Crow,C001121,Democratic,House,Colorado,6.0,D


In [13]:
# State name → USPS abbreviations

state_abbr = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO',
    'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ',
    'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH',
    'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT',
    'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY', 'District of Columbia':'DC'

}

meta_data['State_Abbr'] = meta_data['State'].str.strip().map(state_abbr)
house_pvi['State_Abbr'] = house_pvi['State'].str.strip().map(state_abbr)
senate_pvi['State_Abbr'] = senate_pvi['State'].str.strip().map(state_abbr)

In [14]:
# Build "State_District" key with zero-padded two digits
def make_sd(df, state_col, dist_col):

    # Convert district strings to numeric, default 1 for non‐numeric (at‐large)
    dist_num = pd.to_numeric(df[dist_col], errors='coerce').fillna(1).replace(0, 1).astype(int)

    # Zero‐pad to two digits
    dist_str = dist_num.apply(lambda d: f"{d:02d}")
    return df[state_col].astype(str) + '-' + dist_str

meta_data['State_District'] = make_sd(meta_data, 'State_Abbr', 'District')
house_pvi['State_District'] = make_sd(house_pvi,  'State_Abbr', 'Number')

In [15]:
house_pvi.head()

,State,Number,Member,Party,2025 Cook PVI,PVI_Party,PVI_Number,State_Abbr,State_District
0,Alabama,1,Barry Moore,R,R+27,R,27,AL,AL-01
1,Alabama,2,Shomari Figures,D,D+5,D,5,AL,AL-02
2,Alabama,3,Mike Rogers,R,R+23,R,23,AL,AL-03
3,Alabama,4,Robert Aderholt,R,R+33,R,33,AL,AL-04
4,Alabama,5,Dale Strong,R,R+15,R,15,AL,AL-05


In [16]:
meta_data.head()

,Name,bioguide_id,Party,Chamber,State,District,Party_Abbr,State_Abbr,State_District
0,Laura Friedman,F000483,Democratic,House,California,30.0,D,CA,CA-30
1,Matt Van Epps,V000139,Republican,House,Tennessee,7.0,R,TN,TN-07
2,Cory A. Booker,B001288,Democratic,Senate,New Jersey,NaN,D,NJ,NJ-01
3,Adam B. Schiff,S001150,Democratic,Senate,California,NaN,D,CA,CA-01
4,Alan Armstrong,A000383,Republican,Senate,Oklahoma,NaN,R,OK,OK-01


### **Merge & Filter for House**

We left-join the PVI onto our `meta_data`, then isolate **House* members only and rename columns for clarity.

In [17]:
# Merge on State_District
temp = meta_data.merge(
    house_pvi[['State_District','PVI_Party','PVI_Number']],
    on='State_District',
    how='left'
)

# Keep only House chamber
source_pvi_house = temp.query("Chamber=='House'").copy()

# Rename for consistency
source_pvi_house.rename(columns={
    'PVI_Party':'Cook_PVI_Party',
    'PVI_Number':'Cook_PVI_Number'
}, inplace=True)

In [18]:
# Flag differences
source_pvi_house['PVI_Party_Diff'] = (
    source_pvi_house['Party_Abbr'] != source_pvi_house['Cook_PVI_Party']
)

# Zero out numeric PVI where party mismatches
mask = source_pvi_house['PVI_Party_Diff']
source_pvi_house.loc[mask, 'Cook_PVI_Number'] = 0

## **Finalize & Export**

Select only the columns needed for Bridge Grades and save to CSV.

In [19]:
final_cols = [
    'Name','Chamber','bioguide_id','Party',
    'Cook_PVI_Party','Cook_PVI_Number','PVI_Party_Diff'
]
source_pvi_house[final_cols].to_csv(
    '/content/drive/Shareddrives/Data Archive/Bridge_Grades/Output_Files/Temp_Files/Temp_M/Temp_M_House_v1.0.0_2026_06_02.csv',
    index=False
)

In [20]:
source_pvi_house[final_cols].head(10)

,Name,Chamber,bioguide_id,Party,Cook_PVI_Party,Cook_PVI_Number,PVI_Party_Diff
0,Laura Friedman,House,F000483,Democratic,D,22.0,False
1,Matt Van Epps,House,V000139,Republican,R,10.0,False
5,Analilia Mejia,House,M001246,Democratic,D,5.0,False
6,Clay Fuller,House,F000485,Republican,R,19.0,False
7,Joaquin Castro,House,C001091,Democratic,D,12.0,False
8,Kristen McDonald Rivet,House,M001237,Democratic,R,0.0,True
9,Jason Crow,House,C001121,Democratic,D,11.0,False
10,Kevin Kiley,House,K000401,Independent,R,0.0,True
11,Brendan F. Boyle,House,B001296,Democratic,D,19.0,False
109,Maxwell Frost,House,F000476,Democratic,D,13.0,False


### **Prepare Senator Metadata & Merge**

We filter our `meta_data` to Senate members, derive the first letter of their party, and merge on state name.

In [21]:
sen_meta = meta_data.query("Chamber=='Senate'").copy()

In [22]:
# Merge PVI onto senators by State
senate_merge = sen_meta.merge(
    senate_pvi[['State','PVI_Party','PVI_Number']],
    on='State',
    how='left'
)

### **Flag & Adjust Party Mismatches**

If a senator’s party letter differs from the Cook‐PVI lean, we zero out the PVI number so they aren’t credited for an opposing lean.

In [23]:
# Flag mismatches between senator’s party and PVI lean
senate_merge['PVI_Party_Diff'] = (
    senate_merge['Party_Abbr'] != senate_merge['PVI_Party']
)

# Zero‐out PVI_Number where mismatch is True
mask = senate_merge['PVI_Party_Diff']
senate_merge.loc[mask, 'PVI_Number'] = 0

### **Finalize & Export**

Select the necessary columns and save the CSV for Source M – Senate.

In [24]:
# Select output fields
final_cols = ['Name','Chamber','bioguide_id','Party','PVI_Party','PVI_Number','PVI_Party_Diff']
source_pvi_senate = senate_merge[final_cols]

# Export
source_pvi_senate.to_csv(
    '/content/drive/Shareddrives/Data Archive/Bridge_Grades/Output_Files/Temp_Files/Temp_M/Temp_M_Senate_v1.0.0_2026_06_02.csv',
    index=False
)

In [25]:
senate_merge[final_cols].head(10)

,Name,Chamber,bioguide_id,Party,PVI_Party,PVI_Number,PVI_Party_Diff
0,Cory A. Booker,Senate,B001288,Democratic,D,4,False
1,Adam B. Schiff,Senate,S001150,Democratic,D,12,False
2,Alan Armstrong,Senate,A000383,Republican,R,17,False
3,Todd Young,Senate,Y000064,Republican,R,9,False
4,Ron Wyden,Senate,W000779,Democratic,D,8,False
5,Roger F. Wicker,Senate,W000437,Republican,R,11,False
6,Sheldon Whitehouse,Senate,W000802,Democratic,D,8,False
7,Peter Welch,Senate,W000800,Democratic,D,17,False
8,Elizabeth Warren,Senate,W000817,Democratic,D,14,False
9,Raphael G. Warnock,Senate,W000790,Democratic,R,0,True
